# 🧬 00 · Welcome to Bioimage Analysis
### From pixels to biological insight — what this course is and how it works

**Part A · Foundations** · Module 00 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/00_welcome_bioimage_analysis.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib |
| **Prerequisites** | None — this is the starting point |
| **Estimated time** | 45–60 minutes |
| **02-740 topics covered** | Course overview; the acquire → process → measure → decide pipeline; historical and interdisciplinary context |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All figures in this notebook are drawn with code — nothing is downloaded, and every number you see comes from a formula or a simulation you can inspect.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. Describe, in your own words, why biologists need *computational* image analysis rather than just looking at pictures.
2. Name the four stages of the bioimage analysis pipeline — **acquire → process → measure → decide** — and give a biological example of each.
3. Recognize why bioimage informatics sits at the intersection of microscopy, computer vision, statistics, and molecular biology.
4. See a roadmap of the 37 modules ahead and understand how they build on one another.

## 🗺️ Contents

1. Why biologists need bioimage analysis
2. The bioimage analysis pipeline: acquire → process → measure → decide
3. A short history, and why this field is interdisciplinary
4. Roadmap of this course

## 🧫 The biological question

Here is a number worth sitting with: in an adult human body, roughly **25 million cell divisions**
happen *every second*. Chromosomes condense, spindle fibers assemble, sister chromatids separate,
membranes pinch closed — an enormous, continuous, three-dimensional choreography, repeated
trillions of times over a lifetime, almost all of it invisible to the naked eye.

Biologists have a remarkable set of tools for *seeing* this choreography. A protein can be tagged
with green fluorescent protein (GFP) — a molecule originally isolated from the jellyfish *Aequorea
victoria* — so that wherever that protein goes in a living cell, it glows green under the right
light. Point a fluorescence microscope at a dividing cell and you can watch the mitotic spindle
form, the chromosomes align, and the two daughter cells pull apart, all in a movie.

But a movie, by itself, is not data. If you want to ask "how fast do the chromosomes separate?" or
"does this drug slow down spindle assembly?" or "are these two proteins colocalized at the same
structure?", eyeballing the video will not get you a number you can put in a figure, run a
statistical test on, or compare across hundreds of cells and dozens of experimental conditions.
**Images have to be turned into measurements.** That conversion — from pixels to quantitative,
reproducible, statistically testable biological insight — is exactly what this course is about.

Three recurring motivating examples will follow you through this course, each a real research
problem:

- **Mitotic spindle dynamics.** Two-channel fluorescence movies of dividing cells (chromosomes in
  one channel, spindle fibers in another) that must be segmented, tracked, and measured frame by
  frame to quantify how division proceeds — and how it goes wrong in cancer.
- **Axonal cargo transport.** Neurons are enormous, thin cells — some axons are a meter long in a
  human body — and they must shuttle vesicles, mitochondria, and proteins between the cell body and
  the far end of the axon. Fluorescently labeled cargo appears in a microscope as small, moving,
  noisy dots that must be detected and tracked to measure transport speed and direction — a process
  directly relevant to diseases like Alzheimer's, where transport breaks down.
- **Actin networks and cell migration.** The actin cytoskeleton pushes the leading edge of a
  migrating cell (for instance, a white blood cell chasing a bacterium) forward. Visualizing and
  quantifying this filamentous, branching network — at both the light-microscope and
  electron-microscope scale — requires very different image-analysis tools depending on how the
  image was captured.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

This module does not need any simulators yet — we will start pulling functions from the shared toolkit beginning in Module 01. For now, all we need is NumPy and Matplotlib, already imported above.

---
## 1 · Why biologists need bioimage analysis

In [ ]:
# ── 1.1 a tiny illustration: "looking" vs "measuring" ──
# Simulate a very simple scenario: a cell's fluorescence intensity over time, as it might look
# in a single pixel of a time-lapse movie (e.g. a reporter for gene expression).
t = np.arange(0, 120)  # minutes
true_signal = 0.4 + 0.5 * (1 / (1 + np.exp(-(t - 60) / 8)))  # a biological switch-like rise
observed = true_signal + rng.normal(0, 0.05, size=t.shape)   # what the camera actually records

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(t, observed, '.', color='gray', alpha=0.6, label='what the camera records (noisy)')
ax.plot(t, true_signal, '-', color='crimson', lw=2, label='the biological process (hidden)')
ax.set_xlabel('time (minutes)'); ax.set_ylabel('fluorescence intensity (a.u.)')
ax.set_title('A biological switch, as seen through a noisy detector')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('Can you tell, just by eye, the exact minute the switch turned on? Decide for yourself before reading on.')

🔍 **What just happened?** We simulated one pixel's worth of a fluorescence time-lapse: a
biological process (a sigmoid "switch", like a gene turning on) hidden underneath realistic camera
noise. Even in this absurdly simple, one-dimensional case, it is hard to read off a precise
"switch-on time" by eye from the noisy dots alone. Real experiments involve thousands of such
pixels, moving in two or three spatial dimensions, across hundreds of cells — eyeballing does not
scale, and it is not reproducible: two people staring at the same movie will not extract the same
numbers. We need algorithms that take an image in and a measurement out, consistently.

✏️ **Your turn.** Change `true_signal` above to a different biological shape — for example a
pulse that goes up and then back down (think of a transient signaling event), or a step function
with no smooth transition. Re-run the cell. Does the noise make it easier or harder to tell the two
shapes apart? This is a preview of a question that will recur throughout the course: *how much
signal do you need, relative to the noise, before a measurement becomes trustworthy?*

### 📘 Lecture notes — what bioimage analysis is

**Definition.** Bioimage analysis uses computational methods to analyze and understand images that
capture biological processes. It is often considered a branch of bioinformatics and computational
biology, and it typically involves four components working together: **low-level image analysis**
(e.g. detecting a feature such as a bright spot), **high-level information extraction** (e.g.
recognizing a pattern or classifying a cell type), **image data management** (organizing and storing
very large image datasets), and **image visualization** (turning results back into pictures humans
can interpret).

**Worked example.** A single modern microscopy experiment can produce many gigabytes of images per
hour. Multiply that by hundreds of experimental conditions, and "just look at it" stops being
physically possible — a computational pipeline is not a luxury, it is the only way the experiment
can be analyzed at all.

**One memorable point.** *Bioimage analysis exists because biological processes are too fast, too
small, too numerous, or too subtle for a human eye — however well trained — to measure reliably and
reproducibly at scale.*

---
## 2 · The bioimage analysis pipeline

Nearly every project in this course, and in the research literature, follows the same four-stage
pipeline:

$$\textbf{Acquire} \;\rightarrow\; \textbf{Process} \;\rightarrow\; \textbf{Measure} \;\rightarrow\; \textbf{Decide}$$

- **Acquire** — capture the image with a microscope (Module 02 covers the basics; Modules 03–10 go deep into the physics and instrumentation behind it: optics, diffraction, aberrations, contrast and fluorescence techniques, optical sectioning, and the camera's own detection chain).
- **Process** — clean up, filter, and enhance the raw image so the signal of interest stands out from noise and artifacts (Modules 11–12, 24–25).
- **Measure** — detect, segment, classify, track, and quantify the biological objects of interest (Modules 13–23, 26).
- **Decide** — use the measurements to test a biological hypothesis, train a model, or make a scientific claim (woven throughout, and the subject of Modules 15–21, 28–36).

*In words:* this is a feedback loop, not a one-way street. A poor acquisition (too much noise, too
coarse a sampling rate) can make the processing and measurement stages impossible no matter how
clever the algorithm — so understanding image formation (Module 02) earns its place as the very
next step after this one.

In [ ]:
# ── 2.1 draw the pipeline as a simple schematic (generated with code, never a stored picture) ──
fig, ax = plt.subplots(figsize=(9, 2.2))
stages = ['Acquire\n(microscope)', 'Process\n(filter/denoise)', 'Measure\n(detect/segment/track)', 'Decide\n(test/model/claim)']
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
for i, (s, c) in enumerate(zip(stages, colors)):
    ax.add_patch(plt.Rectangle((i * 2.2, 0), 1.8, 1, color=c, alpha=0.85))
    ax.text(i * 2.2 + 0.9, 0.5, s, ha='center', va='center', color='white', fontsize=9, fontweight='bold')
    if i < len(stages) - 1:
        ax.annotate('', xy=(i * 2.2 + 2.15, 0.5), xytext=(i * 2.2 + 1.85, 0.5),
                    arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.set_xlim(-0.3, 4 * 2.2); ax.set_ylim(-0.3, 1.3); ax.axis('off')
ax.set_title('The bioimage analysis pipeline')
plt.tight_layout(); plt.show()

🔍 **What just happened?** We drew the pipeline as a simple flow diagram — purely with
Matplotlib shapes and text, no stored image file. Every figure in this entire course is built this
way: generated fresh from code, so it can be regenerated, modified, and inspected rather than taken
on faith.

---
## 3 · A short history, and why this field is interdisciplinary

The light microscope is more than 300 years old — Robert Hooke first used one in 1665 to look at a
thin slice of cork, and coined the word *cell* for the small box-like compartments he saw. For most
of those three centuries, microscopy was a visual, qualitative science: a trained eye looking
through an eyepiece.

That changed starting in the early 1980s, when digital image processing began to be applied
systematically to microscopy, pioneered by researchers like Shinya Inoue, who combined video
cameras with computational image enhancement to reveal cellular structures that were otherwise
invisible. From there, several developments in biology — genomics and proteomics, systems biology,
and high-throughput screening — created a flood of large, complex image datasets that could no
longer be analyzed one picture at a time by a person, giving rise to the field now called
**bioimage informatics**.

Because of this history, bioimage informatics pulls together:

- **microscopy** (especially fluorescence and electron microscopy) — how the image was formed in the first place;
- **image processing and computer vision** — extracting meaningful structure from pixels;
- **statistics, pattern recognition, and machine learning** — turning structure into biological conclusions;
- **databases, information retrieval, and distributed computation** — managing and scaling to very large datasets;
- **molecular biology, biochemistry, genetics, and biophysics** — the actual scientific questions driving all of the above.

No single person is an expert in all five. Expect to borrow vocabulary from fields you have not
studied yet — this course will introduce each piece from scratch as it becomes relevant.

### 📘 Lecture notes — why this field is interdisciplinary

**Definition.** Bioimage data poses challenges that are distinct from ordinary photography: very
large data volumes, highly dynamic and complex biological events, and a hard requirement that
algorithms be robust (because the data is noisy) and efficient (because there is so much of it).

**Worked example.** Light microscopy and electron microscopy sit at opposite ends of a resolution
trade-off: light microscopy can resolve structures larger than roughly 200 nanometers and can image
*living* cells, because visible light does little damage; electron microscopy can resolve structures
down to about 0.1 nanometers, but requires the sample to be fixed (chemically preserved) and placed
in a vacuum, since electrons are both very damaging to live tissue and cannot travel through air.
Choosing a microscopy technique is itself a trade-off between resolution and the ability to observe
a live, dynamic process.

**One memorable point.** *The choice of imaging technique and the choice of analysis algorithm are
not independent decisions — the same biological question can become dramatically easier or harder
to answer depending on how the data was collected in the first place.*

---
## 4 · Roadmap of this course

In [ ]:
# ── 4.1 draw the 37-module roadmap, grouped into parts ──
parts = [
    ('A', 'Foundations', 2, '#4C72B0'),
    ('B', 'Seeing like a microscope', 11, '#4C72B0'),
    ('C', 'Finding structure', 2, '#55A868'),
    ('D', 'Classifying bioimages', 2, '#55A868'),
    ('E', 'Latent structure', 1, '#55A868'),
    ('F', 'Segmenting, detecting, synthesizing', 4, '#55A868'),
    ('G', 'Across time and space', 2, '#C44E52'),
    ('H', 'Restoring signal', 2, '#C44E52'),
    ('I', 'Quantifying & modern architectures', 2, '#C44E52'),
    ('J', 'Research frontier', 8, '#8172B2'),
    ('K', 'Capstone', 1, '#333333'),
]
fig, ax = plt.subplots(figsize=(10, 4.5))
y = 0
for letter, name, n_modules, color in parts:
    ax.barh(y, n_modules, color=color, alpha=0.85)
    ax.text(n_modules + 0.2, y, f'Part {letter} · {name} ({n_modules} modules)', va='center', fontsize=8.5)
    y += 1
ax.set_ylim(-0.6, y - 0.4); ax.invert_yaxis()
ax.set_xlabel('number of modules'); ax.set_yticks([])
ax.set_title('Roadmap: 37 modules across 11 parts')
plt.tight_layout(); plt.show()
print('Level 1 (Modules 00-27): core bioimage analysis, grounded in classical and modern methods.')
print('Level 2 (Modules 28-35): the research frontier — topics at the cutting edge of the field today.')
print('Capstone (Module 36): a complete pipeline, built end to end, from raw simulated data to a biological conclusion.')

🔍 **What just happened?** We visualized how the 37 modules are organized: roughly two-thirds
build up the "classical-to-modern" foundations of bioimage analysis (Parts A–I), while the last
third (Part J) pushes out to problems that are active areas of research right now — the kind of
work you would find in a current conference paper rather than a decade-old textbook. Each module
builds on specific earlier ones; look for the "Builds on" line in later modules' key-information
tables (Level 2 modules name a "Builds on" field explicitly).

✏️ **Your turn.** Looking at the roadmap figure, pick one "Part" whose topic you are most curious
about, and write one sentence (in a comment, or just to yourself) predicting what biological
question you think it will help answer. There is no wrong answer — the point is to arrive at that
module later with a question already in mind.

---
## 📌 Key takeaways

- Biological processes happen too fast, too small, and in too much volume for manual visual inspection to produce reliable, reproducible measurements — this is the entire motivation for the field.
- Every project in this course follows the same four-stage pipeline: **acquire → process → measure → decide**.
- Bioimage informatics is genuinely interdisciplinary: microscopy, computer vision, statistics and machine learning, informatics, and molecular biology all contribute.
- This course has 37 modules: 28 core modules (00–27) grounded in established methods, 8 research-frontier modules (28–35), and one capstone (36) that ties everything together.

## ✅ Checkpoint

**1.** Why is "just looking at the movie" not sufficient for most bioimage analysis tasks?

<details><summary>Answer</summary>
Because measurements need to be precise, reproducible across many samples and observers, and
quantitative enough to support statistical testing — all things that visual inspection alone cannot
reliably provide, especially at the scale (many cells, many conditions, many time points) at which
modern experiments operate.
</details>

**2.** Put the four pipeline stages in order and give one example biological task for each.

<details><summary>Answer</summary>
Acquire (capture a fluorescence movie of a dividing cell) → Process (denoise/filter the movie) →
Measure (segment and track the chromosomes) → Decide (test whether a drug changes the separation
speed).
</details>

**3.** Give one advantage and one disadvantage of electron microscopy compared to light microscopy.

<details><summary>Answer</summary>
Advantage: far higher resolution (down to about 0.1 nm vs. about 200 nm), revealing subcellular
detail invisible to light microscopy. Disadvantage: samples must be fixed (chemically preserved) and
imaged in a vacuum, so living, dynamic processes cannot be observed.
</details>

## 🧠 Questions & discussion

- If you could design the "perfect" imaging experiment for a biological question you personally
  find interesting, what would you want to measure, and at what spatial and temporal resolution?
- Why might a biologist and a computer scientist studying the exact same microscope image focus on
  completely different aspects of it?
- The course roadmap devotes 8 of 37 modules to "research frontier" topics. Why do you think a
  foundational course would deliberately include material that is still actively changing?

## 🚀 Bonus — going further

- Look up the 2008 Nobel Prize in Chemistry (awarded for the discovery and development of GFP) and
  read a short account of how GFP was discovered and why it mattered.
- Watch (search for) a real time-lapse fluorescence movie of mitosis or axonal transport online, and
  try to describe in words what pipeline stage (acquire/process/measure/decide) each part of what
  you are watching corresponds to.
- Skim the Wikipedia article on "Bioimage informatics" for a few more historical examples.

## 📝 Lab note

```text
Module: 00 — Welcome to Bioimage Analysis
Date:
One thing I learned:
One question I still have:
One biological process I would like to image and measure, if I could:
```

## 📚 References

1. Shimomura, O., Johnson, F.H., Saiga, Y. (1962). Extraction, purification and properties of
   aequorin. *Journal of Cellular and Comparative Physiology*, 59(3), 223–239.
2. Tsien, R.Y. (1998). The green fluorescent protein. *Annual Review of Biochemistry*, 67, 509–544.
3. Inoué, S. (1986). *Video Microscopy*. Plenum Press.
4. Peng, H. (2008). Bioimage informatics: a new area of engineering biology. *Bioinformatics*,
   24(17), 1827–1836.
5. Pepperkok, R., Ellenberg, J. (2006). High-throughput fluorescence microscopy for systems biology.
   *Nature Reviews Molecular Cell Biology*, 7, 690–696.
6. Walter, T. et al. (2010). Visualization of image data from cells to organisms. *Nature Methods*,
   7, S26–S41.
7. Meijering, E., Carpenter, A.E., Peng, H., Hamprecht, F.A., Olivo-Marin, J.C. (2016). Imagining the
   future of bioimage analysis. *Nature Biotechnology*, 34, 1250–1255.

## ➡️ Next

Module 01 gets you fluent in the one tool every subsequent module relies on: representing and
manipulating images as NumPy arrays in Python. No prior programming experience is assumed — we
start from the very first `import numpy as np`.